# Ejercicio 3: Problema de la Mochila 0/1

**Problema:** seleccionar un subconjunto de n=20 ítems que maximice el valor total sujeto a la restricción de capacidad W=400 kg.

**Representación:** binario directo. El cromosoma `x = [x1,...,x20]` con `xi ∈ {0,1}` ya es la solución — sin decodificación intermedia. A diferencia del ej1 donde el binario codificaba un real vía Gray code, acá el genotipo es idéntico al fenotipo.

In [50]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/datos_mochila.csv")
valores = df['Valor'].to_numpy()
pesos   = df['Peso_kg'].to_numpy()
W = 400
N = len(valores)  # 20

display(df)

,ID,Objeto,Valor,Peso_kg
0,1,Laptop Pro,150,15
1,2,Generador Eléctrico,450,80
2,3,Kit de Herramientas,60,12
3,4,Panel Solar,200,40
4,5,Batería Litio,180,35
5,6,Monitor Curvo,70,20
6,7,Drone de Inspección,220,18
7,8,Caja de Repuestos,120,25
8,9,Impresora 3D,300,45
9,10,Kit Médico,110,10


## Operador de reparación y fitness

Usamos un **repair operator** que solo hace factible al individuo, sin modificar soluciones ya factibles:
- Si el peso supera W: eliminar ítems con el peor ratio valor/peso hasta ser factible.

No agregamos ítems extra porque eso convertiría toda la población a la misma solución greedy, eliminando la diversidad necesaria para que el GA funcione.

In [51]:
def repair(ind, valores, pesos, W):
    ind = ind.copy()
    ratios = valores / pesos
    # eliminar ítems con peor ratio hasta ser factible
    while pesos[ind == 1].sum() > W:
        selected = np.where(ind == 1)[0]
        worst = selected[np.argmin(ratios[selected])]
        ind[worst] = 0
    return ind


def fitness_backpack(ind, valores, pesos, W):
    ind = repair(ind, valores, pesos, W)
    return float(valores[ind == 1].sum())

## Operadores genéticos

Al trabajar con binario directo, los operadores estándar funcionan sin modificaciones:
- **Crossover de 2 puntos** (versión simplificada sin split x/y del ej1)
- **Mutación por bit-flip**: cada bit se flippea con probabilidad pm

In [52]:
def crossover_1d(p1, p2, pc):
    if np.random.rand() >= pc:
        return p1.copy(), p2.copy()
    a, b = sorted(np.random.choice(len(p1), 2, replace=False))
    c1 = np.concatenate([p1[:a], p2[a:b], p1[b:]])
    c2 = np.concatenate([p2[:a], p1[a:b], p2[b:]])
    return c1, c2


def mutate_flip(ind, pm):
    ind = ind.copy()
    mask = np.random.rand(len(ind)) < pm
    ind[mask] = 1 - ind[mask]
    return ind


def selection_windows(pop_size, fitness, population):
    idx_sorted = np.argsort(fitness)  
    selected_indices = []
    for i in range(pop_size):
        window_size = max(1, int(pop_size * (1 - i / pop_size)))
        pick = np.random.randint(0, window_size)
        selected_indices.append(idx_sorted[pick])
    return population[selected_indices]


def get_elites(population, fitness, n_elites):
    elite_indices = np.argsort(fitness)[:n_elites]
    return population[elite_indices].copy()

## Algoritmo Genético

El problema es de **maximización**, pero `selection_windows` está diseñada para minimización. Se resuelve pasando `-fitness_vals` como argumento.

In [53]:
def run_backpack_ga(pop_size, n_gen, pc, pm, n_elites, valores, pesos, W):
    n = len(valores)
    population = np.random.randint(0, 2, (pop_size, n))
    history = []

    for gen in range(n_gen):
        fitness_vals = np.array([fitness_backpack(ind, valores, pesos, W) for ind in population])
        neg_fitness = -fitness_vals  # invertir para que selection_windows funcione (maximizacion -> minimizacion)

        history.append(np.max(fitness_vals))

        new_population = list(get_elites(population, neg_fitness, n_elites))
        mating_pool = selection_windows(pop_size, neg_fitness, population)

        idx = 0
        while len(new_population) < pop_size:
            p1 = mating_pool[idx % pop_size]
            p2 = mating_pool[(idx + 1) % pop_size]
            c1, c2 = crossover_1d(p1, p2, pc)
            new_population.append(mutate_flip(c1, pm))
            if len(new_population) < pop_size:
                new_population.append(mutate_flip(c2, pm))
            idx += 2

        population = np.array(new_population)

    # extraer mejor individuo final
    fitness_vals = np.array([fitness_backpack(ind, valores, pesos, W) for ind in population])
    best_ind = repair(population[np.argmax(fitness_vals)], valores, pesos, W)

    return best_ind, history

In [54]:
best_ind, history = run_backpack_ga(
    pop_size=20,   # poblacion chica: obliga al GA a trabajar mas generaciones
    n_gen=100,
    pc=0.85,
    pm=0.15,       # ~3 bits flipeados por individuo: suficiente para explorar con n=20
    n_elites=1,
    valores=valores, pesos=pesos, W=W
)

print(f"Valor total:  {valores[best_ind == 1].sum()}")
print(f"Peso total:   {pesos[best_ind == 1].sum()} kg  (limite: {W} kg)")
print()
print("Items seleccionados:")
display(df[best_ind == 1][['Objeto', 'Valor', 'Peso_kg']])

Valor total:  2870
Peso total:   398 kg  (limite: 400 kg)

Items seleccionados:


,Objeto,Valor,Peso_kg
0,Laptop Pro,150,15
1,Generador Eléctrico,450,80
2,Kit de Herramientas,60,12
3,Panel Solar,200,40
4,Batería Litio,180,35
6,Drone de Inspección,220,18
8,Impresora 3D,300,45
9,Kit Médico,110,10
10,Unidad de Almacenamiento,80,12
13,Escáner Láser,310,30


## Solución óptima por Programación Dinámica

Con n=20 y W=400, la DP tiene complejidad O(n·W) = O(8000) — trivial. Nos da el **óptimo exacto** para medir el gap del AG.

In [55]:
def backpack_dp(valores, pesos, W):
    n = len(valores)
    dp = np.zeros((n + 1, W + 1), dtype=int)
    for i in range(1, n + 1):
        for w in range(W + 1):
            dp[i, w] = dp[i-1, w]
            if pesos[i-1] <= w:
                dp[i, w] = max(dp[i, w], dp[i-1, w - pesos[i-1]] + valores[i-1])
    return dp[n, W]


optimo = backpack_dp(valores, pesos, W)
valor_ga = int(valores[best_ind == 1].sum())
gap = (optimo - valor_ga) / optimo * 100

print(f"Optimo (DP):  {optimo}")
print(f"AG:           {valor_ga}")
print(f"Gap:          {gap:.2f}%")

Optimo (DP):  2870
AG:           2870
Gap:          0.00%


## Visualización

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Curva de convergencia
axes[0].plot(history, color='steelblue', linewidth=2)
axes[0].axhline(optimo, color='tomato', linestyle='--', label=f'Optimo DP = {optimo}')
axes[0].set_xlabel('Generacion')
axes[0].set_ylabel('Mejor valor encontrado')
axes[0].set_title('Convergencia del AG - Mochila 0/1')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Todos los items: seleccionados vs descartados
df_result = df.copy()
df_result['Seleccionado'] = best_ind
df_result = df_result.sort_values('Seleccionado', ascending=False)

colores = ['steelblue' if s == 1 else 'lightgray' for s in df_result['Seleccionado']]
bars = axes[1].barh(df_result['Objeto'], df_result['Valor'], color=colores)

# Etiqueta de peso en cada barra
for bar, (_, row) in zip(bars, df_result.iterrows()):
    axes[1].text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
                 f"{row['Peso_kg']} kg", va='center', fontsize=8,
                 color='dimgray')

axes[1].set_xlabel('Valor')
axes[1].set_title('Seleccionados (azul) vs Descartados (gris)')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)
axes[1].set_xlim(0, df_result['Valor'].max() * 1.18)

# Linea divisoria entre seleccionados y descartados
n_sel = best_ind.sum()
axes[1].axhline(n_sel - 0.5, color='tomato', linestyle='--', linewidth=1)

plt.tight_layout()
plt.show()

# Tablas separadas
print(f"\n{'='*45}")
print(f"  EN LA MOCHILA  ({n_sel} items, {pesos[best_ind==1].sum()} kg, valor {valores[best_ind==1].sum()})")
print(f"{'='*45}")
display(df[best_ind == 1][['Objeto', 'Valor', 'Peso_kg']].reset_index(drop=True))

print(f"\n{'='*45}")
print(f"  FUERA DE LA MOCHILA  ({(1-best_ind).sum()} items)")
print(f"{'='*45}")
display(df[best_ind == 0][['Objeto', 'Valor', 'Peso_kg']].reset_index(drop=True))